# BT4012 Illicit Bitcoin Transaction Detection

Model: histogram gradient boosting, "lr05" settings (raw features +
in/out degree, no sample weighting), plus a post-processing rule for "quiet" test steps.

Train/test consistency: in the test period only labelled transactions carry features.
Every new feature is computed among feature-bearing labelled + test transactions only
(mask M), so it means the same thing in both periods. No labels are used in any feature.

Outputs: OUT_BEST (quiet-step weight 0.35, final pick 1) and OUT_HEDGE (weight 0.2,
final pick 2.) The engineered
features are kept in the validation comparison; the final model does not use them.
Structure: 1 config | 2 load | 3 graph | 4 features | 5 model + validation | 6 final fit

## 1. Configuration


In [ ]:
import os
import time
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(max(1, (os.cpu_count() or 2) - 1)))

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path(".")
RUN_CV = True
OUT_BEST = "submission_lr05_quietinv35.csv"    # public 0.95566
OUT_HEDGE = "submission_lr05_quietinv20.csv"   # public 0.95359

# long-horizon forward-chaining folds (fold 0 is the
# most test-like). EASY_FOLDS validate on steps whose illicit patterns were mostly seen in
# training 
FOLDS = [
    (range(1, 22), range(22, 36)),
    (range(1, 19), range(19, 33)),
    (range(1, 15), range(15, 29)),
]
EASY_FOLDS = [(range(1, 29), range(29, 36)), (range(1, 25), range(25, 36))]
CV_SEEDS = (3, 4, 5)
FINAL_SEEDS = (42, 43, 44, 45, 46)


HGB_PARAMS = dict(learning_rate=0.05, max_iter=150, max_leaf_nodes=15, min_samples_leaf=60,
                  max_features=0.5, l2_regularization=1.0, early_stopping=False)
KS = (10, 20, 30, 50, 100)   # neighbour ranks used for crowding distances

# Quiet-step rule: a step is quiet when fewer than QUIET_FLAG_SHARE of its rows score above
# 0.5 (decided from the model's own scores, not the step number). In quiet steps the
# ranking becomes (1 - w) * model rank + w * rank of (minus crowding).
QUIET_FLAG_SHARE = 0.01
QUIET_WEIGHT = 0.35
QUIET_WEIGHT_HEDGE = 0.2

## 2. Load data and base features

Reads the training, test and edge files and stacks training and test transactions into one table, so every feature is built the same way for both. Adds in-degree and out-degree from the full edge list, and defines the masks used throughout: labelled training rows, test rows, and **M** (transactions that carry features, i.e. labelled or test).

In [ ]:
t0 = time.time()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
FEAT_COLS = [c for c in train.columns if c.startswith("feat_")]
LOCAL = list(range(93))             # feat_1..feat_93: the transaction's own (local) features
ALLF = list(range(len(FEAT_COLS)))  # all 165 (local + neighbourhood aggregates)

nodes = pd.concat(
    [train.assign(is_test=False), test.drop(columns="index").assign(is_test=True)],
    ignore_index=True,
).set_index("txId")
N = len(nodes)

# Degree from the full edge list (test-period nodes without features still appear in it).
in_deg = edges["txId2"].value_counts().reindex(nodes.index).fillna(0).values
out_deg = edges["txId1"].value_counts().reindex(nodes.index).fillna(0).values
XB = np.c_[nodes[FEAT_COLS].values, in_deg, out_deg].astype(np.float32)

LAB = (nodes["label"].notna() & ~nodes["is_test"]).values   # labelled training rows
TST = nodes["is_test"].values
M = LAB | TST   # transactions that carry features in the test-period sense 
Y = nodes["label"].fillna(-1).astype(int).values
T = nodes["time_step"].values
print(f"loaded: {N} nodes, {LAB.sum()} labelled, {TST.sum()} test ({time.time() - t0:.0f}s)", flush=True)

loaded: 157101 nodes, 31235 labelled, 15329 test (3s)


## 3. Build the transaction graph

Maps each edge to row positions and keeps only edges whose two ends both have features. An edge A → B means an output of A was spent by B.

In [3]:
pos = pd.Series(np.arange(N), index=nodes.index)
e1 = pos.reindex(edges["txId1"]).values
e2 = pos.reindex(edges["txId2"]).values
ok = ~(np.isnan(e1) | np.isnan(e2))
E1, E2 = e1[ok].astype(int), e2[ok].astype(int)   # E1 -> E2: an output of E1 is spent by E2

## 4. Feature engineering

Three label-free feature groups, each computed only among transactions in **M**, so they mean the same thing in training and test:

- **Chain shape**: component size, links in and out, and longest chains in the graph of feature-bearing transactions.
- **Crowding**: distance to the 10th–100th most similar transaction in the same time step (many near-copies suggest automated activity).
- **Neighbour similarity**: how alike a transaction is to the ones it receives from and pays into.



In [ ]:
# Shape of the subgraph induced by M: component size, in/out degree inside it,
# longest directed path ending at / starting from the node, the longest path in its
# component, and the share of chain-like nodes (in <= 1 and out <= 1) in the component.
def chain_features():
    m = M[E1] & M[E2]
    a, b = E1[m], E2[m]
    succ = sp.csr_matrix((np.ones(len(a)), (a, b)), shape=(N, N))
    pred = succ.T.tocsr()
    _, comp = connected_components(succ, directed=False)
    size = np.bincount(comp)[comp].astype(float)
    din = np.bincount(b, minlength=N).astype(float)
    dout = np.bincount(a, minlength=N).astype(float)
    # Longest paths by dynamic programming over a topological order (the graph is
    # acyclic: a transaction can only spend outputs that already exist).
    indeg, order = din.copy(), []
    stack = list(np.where(M & (indeg == 0))[0])
    while stack:
        v = stack.pop()
        order.append(v)
        for w in succ.indices[succ.indptr[v]:succ.indptr[v + 1]]:
            indeg[w] -= 1
            if indeg[w] == 0:
                stack.append(w)
    up, down = np.zeros(N), np.zeros(N)
    for v in order:
        p = pred.indices[pred.indptr[v]:pred.indptr[v + 1]]
        if len(p):
            up[v] = up[p].max() + 1
    for v in reversed(order):
        s = succ.indices[succ.indptr[v]:succ.indptr[v + 1]]
        if len(s):
            down[v] = down[s].max() + 1
    chainy = ((din <= 1) & (dout <= 1)).astype(float)
    comp_chain = pd.Series(chainy[M]).groupby(comp[M]).mean().reindex(comp).values
    comp_longest = pd.Series((up + down)[M]).groupby(comp[M]).max().reindex(comp).values
    F = np.c_[size, din, dout, up, down, up + down, comp_longest, comp_chain].astype(np.float32)
    F[~M] = np.nan
    return F


"""
Within-step crowding: distance to the k-th most similar transaction of the same
time step (k in KS) plus the mean distance to the 20 and 100 most similar ones.
Features are standardised (clipped at +-50) before measuring distance.
"""
def crowding_features(cols, metric):
    Z = StandardScaler().fit_transform(np.clip(XB[:, cols], -50, 50)).astype(np.float32)
    F = np.full((N, len(KS) + 2), np.nan, dtype=np.float32)
    for s in np.unique(T[M]):
        idx = np.where(M & (T == s))[0]
        k = min(max(KS) + 1, len(idx))
        d, _ = NearestNeighbors(n_neighbors=k, metric=metric).fit(Z[idx]).kneighbors(Z[idx])
        F[idx, :len(KS)] = np.c_[tuple(d[:, min(q, k - 1)] for q in KS)]   # d[:, 0] is the node itself
        F[idx, len(KS)] = d[:, 1:min(21, k)].mean(1)
        F[idx, len(KS) + 1] = d[:, 1:k].mean(1)
    return F

"""Min / mean distance (local features) to in- and out-neighbours inside M:consecutive hops of an automated chain look alike."""
def neighbour_similarity():
    Z = StandardScaler().fit_transform(np.clip(XB[:, LOCAL], -50, 50)).astype(np.float32)
    m = M[E1] & M[E2]
    a, b = E1[m], E2[m]
    d = np.linalg.norm(Z[a] - Z[b], axis=1)
    cols = []
    for key in (b, a):   # in-neighbours, then out-neighbours
        g = pd.Series(d).groupby(key)
        cols += [g.min().reindex(range(N)).values, g.mean().reindex(range(N)).values]
    F = np.c_[tuple(cols)].astype(np.float32)
    F[~M] = np.nan
    return F


CROWD_ALL_MAN = crowding_features(ALLF, "manhattan")
X = np.c_[XB,
          chain_features(),
          CROWD_ALL_MAN,
          crowding_features(ALLF, "euclidean"),
          crowding_features(LOCAL, "manhattan"),
          neighbour_similarity()].astype(np.float32)
print(f"feature matrix {X.shape} ({time.time() - t0:.0f}s)", flush=True)

"""
Label-free crowding score: higher = more crowded. Rank-sum of the distance to the
50th neighbour and the mean distance to the 100 nearest (all features, Manhattan).
"""
def crowd_score(rows):
    knn50 = CROWD_ALL_MAN[rows, KS.index(50)]
    mean100 = CROWD_ALL_MAN[rows, len(KS) + 1]
    return -(rankdata(knn50) + rankdata(mean100))


def rank01(p):
    return rankdata(p) / len(p)

feature matrix (157101, 200) (10s)


## 5. Model, quiet-step rule and validation

Defines the seed-averaged gradient-boosting model and the quiet-step rule: a step is *quiet* when fewer than 1% of its transactions score above 0.5, and in quiet steps crowded transactions are pushed down in the ranking (weight 0.35). Then validates on the main and easy folds, comparing the baseline, the model with engineered features, and the crowding score alone. The quiet-step rule cannot be validated here because no validation step is quiet.

In [ ]:
# Average of HGB models trained with different seeds (removes seed noise)
def fit_predict(Xm, train_mask, pred_mask, seeds):
    return np.mean([HistGradientBoostingClassifier(**HGB_PARAMS, random_state=s)
                    .fit(Xm[train_mask], Y[train_mask])
                    .predict_proba(Xm[pred_mask])[:, 1] for s in seeds], axis=0)

#Boolean per row: is the row's time step one where the model flags almost nothing?
def quiet_steps(p, rows):
    return pd.Series(p > 0.5).groupby(T[rows]).transform("mean").values < QUIET_FLAG_SHARE


def quiet_rule(p, rows, weight=QUIET_WEIGHT):
    """In quiet steps, push crowded transactions down; elsewhere keep the model ranking."""
    w = np.where(quiet_steps(p, rows), weight, 0.0)
    return (1 - w) * rank01(p) + w * rank01(-crowd_score(rows))


if RUN_CV:
    # No validation step is quiet, so the quiet rule cannot be validated here: CV compares
    # the two models only. The leaderboard results for the rule are in the header.
    rows = []
    for kind, folds in (("main", FOLDS), ("easy", EASY_FOLDS)):
        for a, b in folds:
            tr, va = LAB & np.isin(T, a), LAB & np.isin(T, b)
            rows.append(dict(folds=kind, train=f"1-{max(a)}", valid=f"{min(b)}-{max(b)}",
                             lr05=roc_auc_score(Y[va], fit_predict(XB, tr, va, CV_SEEDS)),
                             new_features=roc_auc_score(Y[va], fit_predict(X, tr, va, CV_SEEDS)),
                             crowd_score_only=roc_auc_score(Y[va], crowd_score(va))))
    cvres = pd.DataFrame(rows)
    print(cvres.round(4).to_string(index=False))
    print("mean over main folds:")
    print(cvres[cvres.folds == "main"][["lr05", "new_features", "crowd_score_only"]].mean().round(4).to_string())
    print(f"({time.time() - t0:.0f}s)", flush=True)

folds train valid   lr05  new_features  crowd_score_only
 main  1-21 22-35 0.9216        0.9300            0.8998
 main  1-18 19-32 0.9186        0.9426            0.9053
 main  1-14 15-28 0.9679        0.9679            0.9142
 easy  1-28 29-35 0.9965        0.9969            0.9120
 easy  1-24 25-35 0.9966        0.9957            0.9037
mean over main folds:
lr05                0.9360
new_features        0.9468
crowd_score_only    0.9064
(33s)


## 6. Final fit and submission files

Trains on all labelled steps 1–35 (five seeds), prints which test steps are quiet, applies the quiet-step rule, checks each file's format, and writes the two submission files: weight 0.35 (final pick 1) and weight 0.2 (final pick 2).

In [6]:
p_lr05 = fit_predict(XB, LAB, TST, FINAL_SEEDS)
print("share of test rows scoring > 0.5, by step:",
      pd.Series(p_lr05 > 0.5).groupby(T[TST]).mean().round(3).to_dict())
print("quiet steps:", sorted(set(T[TST][quiet_steps(p_lr05, TST)])))

outputs = [(OUT_BEST, quiet_rule(p_lr05, TST, QUIET_WEIGHT)),
           (OUT_HEDGE, quiet_rule(p_lr05, TST, QUIET_WEIGHT_HEDGE))]
for path, target in outputs:
    sub = pd.DataFrame({"index": test["index"].values, "target": target})
    assert len(sub) == len(test) and sub["target"].between(0, 1).all() and sub["index"].is_unique
    sub.to_csv(path, index=False)
    print(f"wrote {path} {sub.shape}")
print(f"done ({time.time() - t0:.0f}s)")

share of test rows scoring > 0.5, by step: {36: 0.02, 37: 0.054, 38: 0.132, 39: 0.062, 40: 0.064, 41: 0.096, 42: 0.09, 43: 0.001, 44: 0.004, 45: 0.004, 46: 0.01, 47: 0.001, 48: 0.0, 49: 0.008}
quiet steps: [43, 44, 45, 46, 47, 48, 49]
wrote submission_lr05_quietinv35.csv (15329, 2)
wrote submission_lr05_quietinv20.csv (15329, 2)
done (38s)
